# 01 — Data assembly

**Mini Project 2.** Reads the 2023 Census totals-by-topic households export,
renames the variable columns using the Stats NZ lookup table, derives the
crowding target, and writes one tidy analysis table — one row per statistical
area 2 (SA2).

Runs top to bottom on a fresh clone. Nothing here is analysis; it is the
plumbing, and it should be dull.

The lookup table is **not joined** to the data. It is used to rename columns,
which is what makes 261 columns called `VAR_4_n` readable.

Downstream notebooks read `outputs/analysis_frame.csv` and
`outputs/frame_meta.json`. The threshold and the crowding definition travel in
the metadata rather than living in someone's memory.


## Setup

Paths, the encoding-tolerant loader, and the measure parser.

`read_measure` and `SENTINEL_FLOOR` now come from `mp2` rather than being
redefined here — the sentinel rule is a property of the Stats NZ source, not of
this notebook, and notebook 02 found a second sentinel (`-997`) that the old
local copy did not catch.


In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

sys.path.append("../src")
import mp2
from mp2 import read_measure, SENTINEL_FLOOR

# The download folder. One place, so the absolute path appears once.
DATA = Path(r"D:\dsai\dsai_projects\Mini-Project-2-Data")
OUT  = mp2.OUT
OUT.mkdir(exist_ok=True, parents=True)

HOUSEHOLDS_FILE = DATA / "2023-census-totals-by-topic-for-households-by-statistical-ar.csv"
LOOKUP_FILE     = DATA / "2023_census_totals_by_topic_for_households_by_sa2_lookup_table.csv"
SA2_SHAPE       = DATA / "statistical-area-2-2023-generalised.shp"

YEAR = mp2.CENSUS_YEAR


def load_raw(path):
    """Read a Stats NZ export. Tries UTF-8, then the common Windows encoding."""
    for enc in ("utf-8-sig", "utf-8", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(path.name)


def slug(value):
    """Lower-case, underscore-separated column name from a Stats NZ label."""
    import re
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


print("data folder   :", DATA)
print("households ok :", HOUSEHOLDS_FILE.exists())
print("lookup ok     :", LOOKUP_FILE.exists())
print("sentinel floor:", SENTINEL_FLOOR)

## Loading and renaming

The lookup file stacks 2013, 2018 and 2023 together, so it is filtered to
2023 and to the `VAR_*` columns before use. Renaming happens on the way in —
every later reference uses the readable name.


In [ ]:
households = load_raw(HOUSEHOLDS_FILE)
lookup     = load_raw(LOOKUP_FILE)

lookup_2023 = lookup[(lookup["Year"] == YEAR) &
                     lookup["Column_name"].astype(str).str.startswith("VAR_")].copy()

lookup_2023["readable_name"] = [
    f"{slug(topic)}_{slug(category)}"
    for topic, category in zip(lookup_2023["Variable1"], lookup_2023["Variable1_category"])
]
rename_columns = dict(zip(lookup_2023["Column_name"], lookup_2023["readable_name"]))

keep = [c for c in rename_columns if c in households.columns]
analysis = (households[["SA22023_V1_00", "SA22023_V1_00_NAME"] + keep]
            .rename(columns={"SA22023_V1_00": "sa2_code",
                             "SA22023_V1_00_NAME": "sa2_name"})
            .rename(columns=rename_columns))

# Coerce every measure column once, here. read_measure returns float64
# unconditionally, so no later .astype("float64") patch is needed.
value_columns = [c for c in analysis.columns if c not in ("sa2_code", "sa2_name")]
analysis[value_columns] = analysis[value_columns].apply(read_measure)

print("rows, columns :", analysis.shape)
print("rate dtype check — all numeric:",
      bool(analysis[value_columns].dtypes.eq("float64").all()))
print("usable SA2 codes:", analysis["sa2_code"].notna().sum(), "of", len(analysis))

## Check the crowding block reconciles

Before trusting the target, verify the crowding index against its own parts.
The index is hierarchical: `crowded` should equal the two "bedrooms needed"
categories combined, and `total` should equal `total stated` plus `not stated`.

Small differences are expected — every count is rounded independently. A
negative total means a sentinel got through, and the assert says so. This is
the check that found the `-999` problem in the first place, so it stays.


In [ ]:
CROWD = {
    "crowded":      "household_crowding_index_crowded",
    "severe":       "household_crowding_index_two_or_more_bedrooms_needed_severely_crowded",
    "moderate":     "household_crowding_index_one_bedroom_needed_crowded",
    "not_crowded":  "household_crowding_index_not_crowded",
    "not_stated":   "household_crowding_index_not_stated",
    "total":        "household_crowding_index_total",
    "total_stated": "household_crowding_index_total_stated",
}

totals = {name: analysis[col].sum() for name, col in CROWD.items()}
for name, value in totals.items():
    print(f"  {name:14s} {value:>12,.0f}")

print()
print(f"crowded  vs severe+moderate : {totals['crowded'] - totals['severe'] - totals['moderate']:+,.0f}")
print(f"total    vs stated+notstated: {totals['total'] - totals['total_stated'] - totals['not_stated']:+,.0f}")

assert totals["crowded"] > 0, "crowded is not positive - is a sentinel getting through?"
assert totals["total"] > 0, "total is negative - a numeric sentinel survived read_measure"
print()
print("No total is negative, and both identities hold to within rounding.")

## The target

**Crowded** means one or more bedrooms needed. `total_stated` is the
denominator — the households that gave a usable answer — because dividing by
`total` would understate the rate wherever non-response is higher.

Note what this target is: the crowding **rate of an area**, not of a household.
Everything modelled downstream is at SA2 level, which is the ecological fallacy
the README names as the leading limitation.

The binary flag is the **top quartile** of the rate. A plain high/low split
would be nearly all negatives, and PR-AUC built on it would be meaningless. The
quartile gives a workable positive class and it matches what the client does
with the result — rank areas for assessment rather than label them.


In [ ]:
# np.nan, never pd.NA: pd.NA propagates object dtype through the division,
# which is what previously produced an object-dtype rate and a quantile that
# only looked correct at four decimal places.
analysis["crowded_n"]     = analysis[CROWD["crowded"]]
analysis["crowding_base"] = analysis[CROWD["total_stated"]]
analysis["crowding_rate"] = (
    analysis["crowded_n"]
    / analysis["crowding_base"].replace(0, np.nan)
).astype("float64")

assert analysis["crowding_rate"].dtype == "float64", analysis["crowding_rate"].dtype
assert not analysis["crowding_rate"].isna().all(), "every rate is missing"
assert analysis["crowding_rate"].dropna().between(0, 1).all(), "rates must sit in 0-1"

# Top-quartile threshold, computed once and written to frame_meta.json so
# notebooks 02-05 reuse it instead of each recalculating.
threshold = float(analysis["crowding_rate"].quantile(0.75))
analysis["crowded_top_quartile"] = (
    analysis["crowding_rate"] >= threshold
).astype("float64")

# Areas with no usable rate are unknown, not negative. Leaving them NaN keeps
# them out of training rather than silently teaching the model they are fine.
analysis.loc[analysis["crowding_rate"].isna(), "crowded_top_quartile"] = np.nan

n_known   = int(analysis["crowding_rate"].notna().sum())
n_positive = int(analysis["crowded_top_quartile"].sum())
n_unknown = int(analysis["crowding_rate"].isna().sum())

print(f"rate dtype             : {analysis['crowding_rate'].dtype}")
print(f"top-quartile threshold : {threshold:.4f}  ({threshold:.2%})")
print()
print(analysis["crowding_rate"].describe().round(5).to_string())
print()
print(f"SA2s with a usable rate: {n_known:,} of {len(analysis):,}")
print(f"without a usable rate  : {n_unknown:,}")
print(f"positive class         : {n_positive:,} SA2s")
print(f"base rate              : {n_positive / n_known:.2%}  (of areas with a rate)")
print()
print(f"predicting the majority class every time scores "
      f"{max(n_positive / n_known, 1 - n_positive / n_known):.2%} accuracy "
      f"and 0.00 recall on the crowded class.")

## Sanity checks

Every one of these should pass. If an assert fires, stop and read the message —
it names the fault.


In [ ]:
assert analysis["sa2_code"].is_unique, "SA2 codes must be unique"
assert analysis["sa2_code"].notna().all(), "no missing SA2 codes"
assert analysis["crowded_n"].min() >= 0, "negative crowding counts"
assert analysis["crowding_rate"].dropna().between(0, 1).all(), "rates must sit in 0-1"

print(f"SA2s                 : {len(analysis):,}")
print(f"unique SA2 codes     : {analysis['sa2_code'].nunique():,}")
print(f"code length          : {sorted(analysis['sa2_code'].astype(str).str.len().unique())}")
print(f"crowded count range  : {analysis['crowded_n'].min():.0f} to {analysis['crowded_n'].max():.0f}")
print(f"crowding rate range  : {analysis['crowding_rate'].min():.4f} to {analysis['crowding_rate'].max():.4f}")
print()
print("All checks passed.")

## Where suppression lands

The README claims suppression concentrates in small areas. This measures it, so
the claim is a finding with a number behind it rather than an assertion.

The count uses `mp2.flag_suppressed`, which covers both sentinels — the old
version counted `== -999` only and so missed `-997` entirely.


In [ ]:
raw = load_raw(HOUSEHOLDS_FILE)
raw_var_columns = [c for c in raw.columns if str(c).startswith("VAR_4_")]

# Count every suppressed or non-applicable cell, not just the -999 sentinel.
analysis["suppressed_n"] = raw[raw_var_columns].apply(mp2.flag_suppressed).sum(axis=1)

lost = analysis["crowding_rate"].isna()

print(f"SA2s without a crowding rate: {lost.sum()} of {len(analysis)}")
print()
print("suppressed cells per SA2:")
print(analysis["suppressed_n"].describe().round(2).to_string())
print()
print(f"mean suppressed cells - SA2s kept : {analysis.loc[~lost, 'suppressed_n'].mean():.1f}")
print(f"mean suppressed cells - SA2s lost : {analysis.loc[lost,  'suppressed_n'].mean():.1f}")
print()
print("suppressed cells in the lost areas  :", int(analysis.loc[lost,  "suppressed_n"].sum()))
print("suppressed cells in all other areas :", int(analysis.loc[~lost, "suppressed_n"].sum()))
print()
print("correlation, suppression vs crowding rate:",
      round(analysis["suppressed_n"].corr(analysis["crowding_rate"]), 3))

## The ranking is noisy at the top

The two areas with the highest crowding rate have three households between
them. This matters for a client who prioritises by ordering, so it is measured
here rather than left as a caveat.


In [ ]:
print("top 10 most crowded SA2s (unsorted by size):")
print(analysis.nlargest(10, "crowding_rate")[
    ["sa2_name", "crowded_n", "crowding_base", "crowding_rate"]
].to_string(index=False))
print()
for cutoff in (20, 50, 100):
    small = analysis["crowding_base"] < cutoff
    print(f"SA2s under {cutoff:>3} households: {small.sum():>3}  "
          f"rate range {analysis.loc[small, 'crowding_rate'].min():.3f} to "
          f"{analysis.loc[small, 'crowding_rate'].max():.3f}")
print()
print("Note the top of the ranking: those areas are not crowded places, they are")
print("tiny denominators. Bowl-of-three areas can score 1.000 and outrank Otara")
print("West at 612 households. Any shortlist needs a size floor.")

## Write the analysis table

`outputs/analysis_frame.csv` is what notebooks 02–05 read. The shapefile and any
geometry stay out of it — notebook 02 reads the boundary file when it needs
maps, and a plain CSV keeps notebooks 03–05 free of the geo stack. SA2 code is
retained as the join key.

`frame_meta.json` records the decisions, so the threshold and the crowding
definition travel with the data.


In [ ]:
# Fail here rather than three notebooks later if a sentinel survived.
mp2.assert_no_sentinels(analysis)

analysis.to_csv(OUT / "analysis_frame.csv", index=False)
print()
print("written:", OUT / "analysis_frame.csv")

frame_meta = {
    "year": YEAR,
    "n_sa2": int(len(analysis)),
    "source": "Stats NZ 2023 Census totals by topic for households, by SA2",
    "licence": "CC BY 4.0, Statistics New Zealand",
    "crowding_definition": "one or more bedrooms needed (crowded)",
    "crowding_denominator": "total stated, excludes not stated",
    "target": "crowded_top_quartile",
    "top_quartile_threshold": threshold,
    "sa2_with_rate": n_known,
    "sa2_without_rate": n_unknown,
    "positive_class": n_positive,
    "base_rate": round(n_positive / n_known, 6),
    "suppressed_cells": int(analysis["suppressed_n"].sum()),
    "sentinel_floor": SENTINEL_FLOOR,
}

(OUT / "frame_meta.json").write_text(json.dumps(frame_meta, indent=2))
print()
print(json.dumps(frame_meta, indent=2))

## Notes for the README

Stats NZ applies fixed random rounding to base 3 and suppresses sensitive cells
below six. Two numeric sentinels appear in the totals-by-topic tables: `-999`
for a suppressed cell and `-997` for a category not collected or not
applicable. Both are mapped to missing by `mp2.read_measure`. Notebook 01's
reconciliation check caught `-999`; notebook 02 caught `-997`, in a column whose
every value was the sentinel — which is why a missing-value count reported zero.

Suppression is heavily concentrated: the median SA2 carries none, while 112
areas (4.7% of the total) average 221 suppressed cells each against 2.7 for the
areas retained. Those 112 areas lose the crowding rate entirely and are excluded
from the model, so the results describe the areas where the census can be
published — systematically the larger ones.
